# ECO346 Midterm: exam notebook

1. In the cell below, type your **registration number** and the **code written on the board**.
2. Run every cell in order (**Runtime → Run all**, or Shift+Enter on each cell).
3. Do not change any other line. Answer on the Google Form, using your printed paper.

In [ ]:
# ---------------- Setup: fill in the two lines below, then run ----------------
REG  = "FA24-BED-___"    # your registration number, e.g. FA24-BED-002
CODE = "_____"           # the code written on the board

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

DATA_URL = "https://raw.githubusercontent.com/rafi-amiruddin/DVTE/main/exam/"
REG = REG.strip().upper().replace(" ", "").replace("-BEDS-", "-BED-")
name = "d_" + hashlib.sha256((REG + CODE.strip().upper()).encode()).hexdigest()[:10] + ".csv"
df = pd.read_csv(DATA_URL + name)

print(len(df), "households loaded for", REG)
df.head()

## Cell C1: summary of monthly spending per person (Rs)

In [ ]:
print(df["pc_exp_month"].describe().round(2))
print()
print("Median:", round(df["pc_exp_month"].median(), 2))

## Cell C2: two histograms of household size (bins of width 2, starting at 0)

In [ ]:
edges = np.arange(0, df["hhsize"].max() + 3, 2)

counts_P = pd.cut(df["hhsize"], edges, right=True, include_lowest=True).value_counts().sort_index()
counts_Q = pd.cut(df["hhsize"], edges, right=False).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, counts, label in [(axes[0], counts_P, "Version P"), (axes[1], counts_Q, "Version Q")]:
    shown = counts[[iv.left < 20 for iv in counts.index]]
    ax.bar(range(len(shown)), shown.values, color="#4a6f8a")
    ax.set_xticks(range(len(shown)))
    ax.set_xticklabels([str(i) for i in shown.index], rotation=45, fontsize=8)
    ax.set_title(label, loc="left")
    ax.set_xlabel("Household size (bin)")
axes[0].set_ylabel("Number of households")
plt.tight_layout()
plt.show()

print("Version P, tallest bar:", counts_P.max(), "households")
print("Version Q, tallest bar:", counts_Q.max(), "households")

## Cell C3: two means of monthly spending per person (Rs)

In [ ]:
mean_1 = df["pc_exp_month"].mean()
mean_2 = np.average(df["pc_exp_month"], weights=df["hhweight"])

print("Mean 1:", round(mean_1, 2))
print("Mean 2:", round(mean_2, 2))

## Cell C4: percentiles of monthly spending per person (Rs)

In [ ]:
p10 = df["pc_exp_month"].quantile(0.10)
p90 = df["pc_exp_month"].quantile(0.90)

print("10th percentile (P10):", round(p10, 2))
print("90th percentile (P90):", round(p90, 2))

## Cell D1: food share and spending per person (Engel's law)

In [ ]:
x  = df["pc_exp_month"]
lx = np.log(x)
y  = df["food_share"]

correlation = np.corrcoef(lx, y)[0, 1]
slope, intercept = np.polyfit(lx, y, 1)

quintile = pd.qcut(x, 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
q_mean_y  = y.groupby(quintile, observed=True).mean()
q_mid_x   = x.groupby(quintile, observed=True).median()
smooth    = lowess(y, lx, frac=0.03)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(x, y, s=6, alpha=0.15, color="0.4")
grid = np.linspace(lx.min(), lx.max(), 100)
ax.plot(np.exp(grid), intercept + slope * grid, color="#C0392B", lw=2.5)                    # curve A
ax.plot(q_mid_x, q_mean_y, color="#1E8449", lw=2.5, marker="o", ms=8)                     # curve B
ax.plot(np.exp(smooth[:, 0]), smooth[:, 1], color="#2471A3", lw=1.5)                       # curve C
ax.text(np.exp(grid[-1]) * 1.05, intercept + slope * grid[-1], "A", color="#C0392B", fontsize=14, weight="bold")
ax.text(q_mid_x.iloc[-1] * 1.25, q_mean_y.iloc[-1], "B", color="#1E8449", fontsize=14, weight="bold")
ax.text(np.exp(smooth[0, 0]) * 0.80, smooth[0, 1] - 0.02, "C", color="#2471A3", fontsize=14, weight="bold")
ax.set_xscale("log")
ax.set_xlabel("Monthly spending per person (Rs, log scale)")
ax.set_ylabel("Food share of spending")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()

print("Correlation between log(spending) and food share:", round(correlation, 6))
print("Slope of the straight line:", round(slope, 6))

## Cell D2: mean food share by expenditure quintile (Q1 poorest, Q5 richest)

In [ ]:
print(q_mean_y.round(6).to_string())

**End of notebook.** Make sure every answer is on the Google Form, then submit or update your response.